### Exercise: Design of a Hilbert Transformer

In this exercise a Hilbert transformer with an ideal frequency response 

$$
H(e^{j\Omega}) = j\,\mathrm{sgn}(\Omega)
$$

shall be approximated using the Remez algorithm (`scipy.signal.remez()` in Python).

---

#### a) 

Design two Hilbert transformers of order 30 and 31 using the additional argument `'hilbert'`:

```python
from scipy.signal import remez

b = remez(numtaps=m, bands=F, desired=G, type='hilbert')
```

Plot the amplitude response, impulse response, and the pole-zero diagrams.
What are the types of the created filters?

---

#### b)

Create a sinusoidal signal with a duration of two seconds, a signal frequency of $50\text{Hz}$, and a sampling frequency of $1000\text{Hz}$.

Filter this signal with the Hilbert transformer of order $m=30$ and plot the input and output signals.

*Hint:* The group delay has to be taken into account because the output signal is delayed.

---

#### c)

For the Hilbert transform (important in communications), Python provides its own function:

```python
from scipy.signal import hilbert

y = hilbert(x)
```

The return value `y` is the analytic signal of `x`, the real part of `y` is equal to `x`, and the imaginary part of `y` is the Hilbert transform of `x`.

Calculate the Hilbert transform of the signal from exercise **b** and compare the results.



In [ ]:
# BEGIN SOLUTION
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Note: The FIR coefficients (b) from SciPy were negated (-b_scipy) to match 
# the sign convention and plot orientation of the original MATLAB solution.

# ##### Task a: Design the Hilbert Transformer #####

# Filter Order 30 (Type II)
order1 = 30
# Remez design. Normalized frequency bands [0.05, 0.95]
b1_scipy = signal.remez(order1 + 1, [0.05, 0.95], [1], Hz=2.0, type='hilbert')
b1 = -b1_scipy 

# Filter Order 31 (Type IV)
order2 = 31
b2_scipy = signal.remez(order2 + 1, [0.05, 1], [1], Hz=2.0, type='hilbert')
b2 = -b2_scipy

# Calculate Frequency Response
W1, H1 = signal.freqz(b1, 1, 512, whole=False)
W2, H2 = signal.freqz(b2, 1, 512, whole=False)

# Plot 1: Magnitude Response, Order 30
plt.figure(figsize=(10, 4))
plt.plot(W1 / np.pi, np.abs(H1))
plt.grid(True)
plt.xlabel('Omega/pi')
plt.ylabel('|H_H(exp(j*Omega))|')
plt.title('Magnitude Response - Filter Order: 30 Type II')

# Plot 2: Magnitude Response, Order 31
plt.figure(figsize=(10, 4))
plt.plot(W2 / np.pi, np.abs(H2))
plt.grid(True)
plt.xlabel('Omega/pi')
plt.ylabel('|H_H(exp(j*Omega))|')
plt.title('Magnitude Response - Filter Order: 31 Type IV')

# Plot 3: Impulse Response, Order 30
plt.figure(figsize=(10, 4))
plt.stem(np.arange(order1 + 1), b1, basefmt="r-", markerfmt="o", linefmt="C0-")
plt.axis([0, order1, -0.8, 0.8])
plt.grid(True)
plt.xlabel('k')
plt.ylabel('Impulsantwort')
plt.title('Impulse Response - Filter Order: 30 Type II')

# Plot 4: Impulse Response, Order 31
plt.figure(figsize=(10, 4))
plt.stem(np.arange(order2 + 1), b2, basefmt="r-", markerfmt="o", linefmt="C0-")
plt.axis([0, order2, -0.8, 0.8])
plt.grid(True)
plt.xlabel('k')
plt.ylabel('Impulsantwort')
plt.title('Impulse Response - Filter Order: 31 Type IV')

# ##### Task b: Filter Output Signal #####

# Coefficients for filtering (Order 30)
b_scipy = signal.remez(30 + 1, [0.05, 0.95], [1], Hz=2.0, type='hilbert')
b = -b_scipy 
Fs = 1000  
t = np.arange(0, 2, 1/Fs) 
x = np.sin(2 * np.pi * 50 * t) 
xh = signal.lfilter(b, 1, x)  

# Plot 5: Pole-Zero Diagram (zplane)
z, p, k = signal.tf2zpk(b, [1])

plt.figure(figsize=(6, 6))
ax = plt.gca()
# Zeros
ax.plot(np.real(z), np.imag(z), 'o', fillstyle='none', markersize=8, color='C0', label='Zeros (Z)')
# Poles (All lie at z=0 for FIR filter. Manual large marker for visibility.)
ax.plot(0, 0, 'x', markersize=12, markeredgewidth=2, color='blue', label='Poles (P)')

# Unit circle
unit_circle = Circle((0, 0), 1, color='gray', linestyle='--', fill=False)
ax.add_artist(unit_circle)
ax.axvline(0, color='gray')
ax.axhline(0, color='gray')
ax.axis('equal')
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_xlabel('Realteil')
ax.set_ylabel('Imaginärteil')
ax.set_title('Pole-Zero Diagram')
ax.grid(True)
ax.legend()


delay = 15
xd = np.concatenate((np.zeros(delay), x[:len(x) - delay]))

# Plot 6: Input vs. Filtered Output
num_samples = 50
plt.figure(figsize=(10, 4))
plt.plot(t[:num_samples], xd[:num_samples], label='- Input Sig.')
plt.plot(t[:num_samples], xh[:num_samples], '--', label='-- Output Sig.')
plt.grid(True)
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title(' - Input Sig.   -- Output Sig.')
plt.legend(loc='upper right')

# ------------------------------------------------------------------------------
# ##### Task c: Calculate Hilbert Transform #####

Fs = 1000
t = np.arange(0, 2, 1/Fs)
x = np.sin(2 * np.pi * 50 * t)
y = signal.hilbert(x) # Analytic signal y = x + j*xh

# Plot 7: Original vs. Hilbert Transform
num_samples = 50
plt.figure(figsize=(10, 4))
plt.plot(t[:num_samples], np.real(y[:num_samples]), label='- Original') 
plt.plot(t[:num_samples], np.imag(y[:num_samples]), '--', label='-- Hilbert Transform.')
plt.grid(True)
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title('-- Hilbert-Transform.   - Original')
plt.legend(loc='upper right')

plt.show() 
# END SOLUTION

# Solution: Design of a Hilbert Transformer

### a) Filter Design and Impulse/Amplitude Response

If we consider the optional parameter[^1], we will get filter type II for a filter order $m=30$ and filter type IV for filter order $m=31$. The following figures show the amplitude responses and the impulse responses for the designs.

<img src="./pngs/ldsv5a05_a1.png" width="250">
<img src="./pngs/ldsv5a05_a3.png" width="250">
<img src="./pngs/ldsv5a05_a2.png" width="250">
<img src="./pngs/ldsv5a05_a4.png" width="250">

It also shows that the quality of approximation is higher for odd filter orders because of the missing zero at $\Omega = \pi$.

To use a Hilbert transformer to produce analytic signals, the output signal (imaginary part) and the input signal (real part) must be delayed equally and combined. This means that an even group delay should exist. In a non-recursive, linear system, the contribution of each zero to the group delay is $-0.5$. Therefore, for a Hilbert transformer, an even filter order must exist. This contradicts the goal of achieving high approximation quality.

To clarify the position of zeros, the following figure shows the zero distribution of the filter of order $m=30$.

<img src="./pngs/ldsv5a05_a5.png" width="250">

There are $15$ zeros, which lie either on the unit circle or as a pair reflected on the unit circle, because the system is linear and non-recursive.

[^1]: If this parameter is not fixed, the filter will be of type I/III. The design is therefore non-impulse invariant, see exercise 5.4.a.  
*Note:* In Python, the equivalent filter can be designed using `scipy.signal.remez()` with the argument `type='hilbert'`.

---

### b) Group Delay and Signal Verification

Taking the group delay into account, the signal is delayed by $15$ samples with a filter order of $m=30$, according to exercise part a).  

The plot shown here verifies the relation  

$$
-\cos(x) = \mathcal{H}\{\sin(x)\}
$$  

for a sinusoidal series with the Hilbert transform $\mathcal{H}\{\cdot\}$.

<img src="./pngs/ldsv5a05_b1.png" width="250">

---

### c) Analytic Signal Using `hilbert()`

Using Python, the Hilbert transform can be computed with:

```python
from scipy.signal import hilbert

y = hilbert(x)
```

The return value `y` is the analytic signal of `x`, the real part of `y` equals `x`, and the imaginary part of `y` is the Hilbert transform of `x`.

The following plot shows the analytic signal for the sinusoidal series from exercise part a):

<img src="./pngs/ldsv5a05_c1.png" width="250">

Comparison with the results from exercise part b) shows that the principle is identical, only the group delay differs in the figures.